# 01 Data loading and sanity checks
We convert the 1.3 GB Kaggle CSV to a compact Parquet file once, draw a 100k-row sample for quick tests, and check the data before any analysis.

**Before running:** select the "Python (ctds)" kernel and run the notebook from the notebooks/ folder (see the README).

In [1]:
# imports
import sys
sys.path.append("..")  # lets us import the src/ folder from the project root

import pandas as pd
from IPython.display import display

from src.config import RAW_CSV, FLIGHTS_PARQUET, TABLES_DIR
from src.data_loading import csv_to_parquet, make_sample, load_flights, compare_dtype_memory
from src.profiling import track, MB

In [2]:
# settings
FORCE = False  # set to True only to rebuild the Parquet file and the sample
pd.set_option("display.float_format", "{:,.3f}".format)


# input files
print("CSV:    ", RAW_CSV, "| exists:", RAW_CSV.exists())
print("Parquet:", FLIGHTS_PARQUET, "| exists:", FLIGHTS_PARQUET.exists())

CSV:     D:\ComputationalTools\data\raw\flight_data_2024.csv | exists: True
Parquet: D:\ComputationalTools\data\processed\flights_2024.parquet | exists: True


## 1. CSV to Parquet

We read the CSV in chunks of 500,000 rows and keep only the columns we need. Each column gets a small type: `category` for the text columns (airline, airports, states and cancellation code), `int8` for the month, day of month, weekday and the cancelled/diverted flags, `Int16` for scheduled clock times (1435 = 14:35), `float32` for minutes and miles, and a datetime for `fl_date`. We never read the actual-time columns, such as `taxi_out` and `air_time`, because they are only known after the flight. No rows are dropped at this step.

In [3]:
# skipped when the Parquet file already exists
conversion_stats = csv_to_parquet(force=FORCE)
conversion_stats

flights_2024.parquet already exists -> skipping (use force=True to redo).


### Memory saved by the smaller types

We load the first 200,000 rows in three ways. All 35 columns with pandas' default types take 65.0 MB, our 23 columns with default types take 41.6 MB, and our 23 columns with the smaller types take 12.0 MB, 5.4 times less than the first. These rows are all from January, but the full year loaded in Section 3 gives the same result: 425 MB for 7 million rows, or 12 MB per 200,000 rows.

In [4]:
# the comparison reads the CSV, so it only runs when the CSV is there
if RAW_CSV.exists():
    dtype_table = compare_dtype_memory(nrows=200_000)
    dtype_table.to_csv(TABLES_DIR / "01_dtype_memory.csv", index=False)
    display(dtype_table)
else:
    print("Raw CSV not found, so the dtype comparison is skipped.")

,setup,columns,memory_mb_for_200000_rows
0,"all 35 columns, default dtypes",35,65.000
1,"our columns, default dtypes",23,41.600
2,"our columns, optimized dtypes",23,12.000


## 2. Development sample (100,000 rows)

We draw 100,000 flights at random with a fixed seed (`RANDOM_SEED = 42`), so everyone in the group gets the same rows. We use the sample only for quick tests; all reported results use the full data.

In [5]:
# skipped when the sample already exists
sample_stats = make_sample(n=100_000, force=FORCE)
sample_stats

flights_2024_sample.parquet already exists -> skipping (use force=True to redo).


## 3. Load and inspect

In [6]:
# load the full year and log the time and memory it takes
path = FLIGHTS_PARQUET
with track("load_parquet", source=path.name) as stats:
    df = load_flights(path)
    stats["rows"] = len(df)


# size of the table in pandas
memory_mb = df.memory_usage(deep=True).sum() / MB

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {path.name}")
print(f"Memory in pandas: {memory_mb:,.1f} MB")

[load_parquet] runtime: 0.63 s | peak memory (Python/pandas): 95.2 MB | peak memory (pyarrow pool): 1035.6 MB
Loaded 7,079,081 rows x 23 columns from flights_2024.parquet
Memory in pandas: 425.4 MB


In [7]:
df.dtypes.to_frame("dtype").T

,fl_date,op_unique_carrier,origin,dest,origin_state_nm,dest_state_nm,cancellation_code,month,day_of_month,day_of_week,...,crs_arr_time,crs_elapsed_time,distance,dep_delay,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
dtype,datetime64[us],category,category,category,category,category,category,int8,int8,int8,...,Int16,float32,float32,float32,float32,float32,float32,float32,float32,float32


In [8]:
df.head()

,fl_date,op_unique_carrier,origin,dest,origin_state_nm,dest_state_nm,cancellation_code,month,day_of_month,day_of_week,...,crs_arr_time,crs_elapsed_time,distance,dep_delay,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024-01-01,9E,JFK,DTW,New York,Michigan,NaN,1,1,1,...,1508,136.000,509.000,-5.000,-19.000,0.000,0.000,0.000,0.000,0.000
1,2024-01-01,9E,MSP,CLE,Minnesota,Ohio,NaN,1,1,1,...,1325,130.000,622.000,-14.000,-30.000,0.000,0.000,0.000,0.000,0.000
2,2024-01-01,9E,JFK,RIC,New York,Virginia,NaN,1,1,1,...,1601,106.000,288.000,-4.000,-20.000,0.000,0.000,0.000,0.000,0.000
3,2024-01-01,9E,RIC,JFK,Virginia,New York,NaN,1,1,1,...,1841,111.000,288.000,-7.000,-42.000,0.000,0.000,0.000,0.000,0.000
4,2024-01-01,9E,DTW,MKE,Michigan,Wisconsin,NaN,1,1,1,...,1034,79.000,237.000,-5.000,-14.000,0.000,0.000,0.000,0.000,0.000


## 4. Sanity checks

Every check is stored in `checks` and saved as one table at the end.

### 4.1 Row count and months

We check that the file covers the whole of 2024 with no missing month, because every later step splits the data by month. If a month were missing, the asserts stop the notebook here.

In [9]:
# every check goes into this dict and is saved as one table at the end
checks = {}


# rows and dates
checks["rows"] = len(df)
checks["first_date"] = df["fl_date"].min().date()
checks["last_date"] = df["fl_date"].max().date()
checks["months_present"] = df["month"].nunique()

print(f"{checks['rows']:,} flights from {checks['first_date']} to {checks['last_date']}")


# stop here if the data is not the full year we expect
assert checks["months_present"] == 12, "some months are missing"
assert str(checks["first_date"]) == "2024-01-01" and str(checks["last_date"]) == "2024-12-31"

7,079,081 flights from 2024-01-01 to 2024-12-31


In [10]:
# flights per month, sorted from January to December
flights_per_month = df["month"].value_counts().sort_index()
flights_per_month.to_frame("flights").T

month,1,2,3,4,5,6,7,8,9,10,11,12
flights,547271,519221,591767,582205,609743,611132,634613,619025,582622,615497,575404,590581


### 4.2 Missing values per column

Most columns have no missing values, and the gaps that exist are expected. `cancellation_code` is only filled for cancelled flights (98.6% missing). `arr_delay` is empty for cancelled and diverted flights (1.6%), and `dep_delay` for most cancelled flights (1.3%), since a diverted flight did leave. The five delay-cause columns are never missing: they are 0 for flights that were not 15+ minutes late (checked in 4.6). Only one flight has no scheduled duration, and the cleaning step in 01c removes it.

In [11]:
# number and share of missing values per column
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "share_missing": df.isna().mean(),
})
missing = missing.sort_values("n_missing", ascending=False)
missing

,n_missing,share_missing
cancellation_code,6982766,0.986
arr_delay,113814,0.016
dep_delay,92970,0.013
crs_elapsed_time,1,0.000
origin,0,0.000
op_unique_carrier,0,0.000
fl_date,0,0.000
dest_state_nm,0,0.000
origin_state_nm,0,0.000
dest,0,0.000


### 4.3 Cancelled and diverted flights

Cancelled (1.4%) and diverted (0.2%) flights have no `arr_delay`. Each flight gets a `status` (completed, diverted or cancelled), so later notebooks can decide which flights to keep.

In [12]:
# status of each flight
# cancelled comes last, so a flight flagged as both counts as cancelled
status = pd.Series("completed", index=df.index)
status[df["diverted"] == 1] = "diverted"
status[df["cancelled"] == 1] = "cancelled"
df["status"] = status.astype("category")


# share of each status
checks["share_cancelled"] = (df["status"] == "cancelled").mean()
checks["share_diverted"] = (df["status"] == "diverted").mean()

status_shares = df["status"].value_counts(normalize=True)
display(status_shares.to_frame("share"))

,share
status,
completed,0.984
cancelled,0.014
diverted,0.002


In [13]:
# cancellation causes
# A = carrier, B = weather, C = National Air System, D = security
cancelled = df[df["status"] == "cancelled"]
cause_shares = cancelled["cancellation_code"].value_counts(normalize=True)
display(cause_shares.to_frame("share of cancellations"))

,share of cancellations
cancellation_code,
B,0.557
A,0.321
C,0.122
D,0.000


In [14]:
# share of flights without arr_delay, for each status
arr_missing = df["arr_delay"].isna()
missing_by_status = arr_missing.groupby(df["status"], observed=True).mean()
display(missing_by_status.to_frame("share arr_delay missing"))


# completed flights should always have an arrival delay
is_completed = df["status"] == "completed"
checks["completed_without_arr_delay"] = int(arr_missing[is_completed].sum())
print("Completed flights without arr_delay:", checks["completed_without_arr_delay"], "(expected: 0 or very few)")

,share arr_delay missing
status,
cancelled,1.000
completed,0.000
diverted,1.000


Completed flights without arr_delay: 0 (expected: 0 or very few)


### 4.4 Arrival delays (completed flights only)

In [15]:
# only completed flights have an arrival delay
completed = df[df["status"] == "completed"]
arr_delays = completed["arr_delay"].dropna()


# delay summary
checks["share_arr_delay_15plus"] = (arr_delays >= 15).mean()
checks["median_arr_delay"] = arr_delays.median()
checks["mean_arr_delay"] = arr_delays.mean()
checks["max_arr_delay"] = arr_delays.max()

print(f"Share of completed flights arriving 15+ min late: {checks['share_arr_delay_15plus']:.1%}")
print("(The proposal's 10k-row sample gave 21.5%, close to the full-data value.)")

Share of completed flights arriving 15+ min late: 20.8%
(The proposal's 10k-row sample gave 21.5%, close to the full-data value.)


In [16]:
# quantiles show how long the tail of extreme delays is
quantiles = arr_delays.quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1])
quantiles.to_frame("arr_delay (min)").T

,0.000,0.010,0.050,0.250,0.500,0.750,0.950,0.990,0.999,1.000
arr_delay (min),-126.000,-37.000,-27.000,-15.000,-6.000,9.000,83.000,215.000,753.000,"3,803.000"


More than half of the flights arrive early: the median delay is −6 minutes. A few extreme delays (753 minutes at the 99.9th percentile, 3,803 at most) pull the mean up to 7 minutes, so we compare clusters and communities with medians and the share of flights 15+ minutes late, not with means.

### 4.5 Airports, carriers and routes

In [17]:
# an airport could appear only as a destination, so we combine both columns
all_airports = set(df["origin"].unique()) | set(df["dest"].unique())

route_cols = ["origin", "dest"]
pair_cols = ["origin", "dest", "op_unique_carrier"]


# counts
checks["n_origin_airports"] = df["origin"].nunique()
checks["n_airports_total"] = len(all_airports)
checks["n_carriers"] = df["op_unique_carrier"].nunique()
checks["n_routes_directed"] = df.groupby(route_cols, observed=True).ngroups
checks["n_route_carrier_pairs"] = df.groupby(pair_cols, observed=True).ngroups

for name in ["n_origin_airports", "n_airports_total", "n_carriers", "n_routes_directed", "n_route_carrier_pairs"]:
    print(f"{name:24s} {checks[name]:>8,}")

n_origin_airports             348
n_airports_total              348
n_carriers                     15
n_routes_directed           6,805
n_route_carrier_pairs      13,258


In [18]:
# flights per route + carrier pair
# RQ1 needs enough flights per pair for a reliable profile
flights_per_pair = df.groupby(pair_cols, observed=True).size()
pair_summary = flights_per_pair.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
pair_summary.to_frame("flights per route+carrier").T

,count,mean,std,min,10%,25%,50%,75%,90%,max
flights per route+carrier,"13,258.000",533.948,688.850,1.000,13.000,81.000,311.500,716.000,"1,366.600","7,622.000"


All 348 airports appear as an origin, so no airport is destination-only. The 13,258 route + carrier pairs are the starting point for RQ1, which keeps only pairs with at least 30 flights in January–June: about 10,000 pairs, which still cover 99% of the January–June flights (see notebook 01b).

### 4.6 Leakage check: do the cause columns sum to `arr_delay`?

The proposal saw this on a 10k-row sample; here we check the full data. If it holds, the cause columns contain the target and can never be used as features.

In [19]:
# delay-cause columns and the late flights they describe
cause_cols = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
late = df[(df["status"] == "completed") & (df["arr_delay"] >= 15)]


# compare the sum of the causes with the arrival delay
# a difference under 0.5 min counts as equal, to allow for rounding
cause_sum = late[cause_cols].sum(axis=1, min_count=1)
difference = (cause_sum - late["arr_delay"]).abs()
not_late_causes = df.loc[df["arr_delay"] < 15, cause_cols]

checks["late_flights_with_causes"] = late[cause_cols].notna().all(axis=1).mean()
checks["causes_sum_to_arr_delay"] = (difference < 0.5).mean()
checks["not_late_with_all_causes_zero"] = (not_late_causes == 0).all(axis=1).mean()


# results
print(f"15+ late flights with all cause columns filled: {checks['late_flights_with_causes']:.2%}")
print(f"15+ late flights whose causes sum to arr_delay: {checks['causes_sum_to_arr_delay']:.2%}")
print(f"Flights < 15 min late with all causes equal to 0: {checks['not_late_with_all_causes_zero']:.2%}")

15+ late flights with all cause columns filled: 100.00%
15+ late flights whose causes sum to arr_delay: 100.00%
Flights < 15 min late with all causes equal to 0: 100.00%


All three checks give 100%: the causes split each late flight's `arr_delay` exactly, so they are outcome columns. We use them only to describe delays (e.g. causes per cluster in RQ1) and never as model features.

### 4.7 Schedule columns (used by RQ1)

These columns are known before departure. Their ranges tell us what the cleaning step must fix; for example, BTS writes midnight as `2400`.

In [20]:
# ranges of the schedule columns
schedule = df[["crs_dep_time", "crs_arr_time", "crs_elapsed_time", "distance"]]
display(schedule.describe().T)

,count,mean,std,min,25%,50%,75%,max
crs_dep_time,"7,079,081.000","1,327.300",493.031,1.000,905.000,"1,320.000","1,736.000","2,400.000"
crs_arr_time,"7,079,081.000","1,491.365",519.438,1.000,"1,103.000","1,517.000","1,925.000","2,359.000"
crs_elapsed_time,"7,079,080.000",146.766,72.387,-160.000,93.000,130.000,177.000,"1,326.000"
distance,"7,079,081.000",833.906,596.254,11.000,399.000,680.000,"1,069.000","5,095.000"


In [21]:
# values the cleaning step has to handle
# "> 0" is False for missing values, so counting the rows where it fails catches both problems
elapsed_ok = df["crs_elapsed_time"] > 0
distance_ok = df["distance"] > 0

checks["crs_dep_time_2400"] = int((df["crs_dep_time"] == 2400).sum())
checks["crs_elapsed_nonpositive_or_missing"] = int(len(df) - elapsed_ok.sum())
checks["distance_nonpositive_or_missing"] = int(len(df) - distance_ok.sum())

print("crs_dep_time == 2400:", checks["crs_dep_time_2400"])
print("crs_elapsed_time <= 0 or missing:", checks["crs_elapsed_nonpositive_or_missing"])
print("distance <= 0 or missing:", checks["distance_nonpositive_or_missing"])

crs_dep_time == 2400: 1
crs_elapsed_time <= 0 or missing: 2
distance <= 0 or missing: 0


## 5. Save the summary

All the checks above go into one table, `results/tables/01_sanity_checks.csv`, with one row per check. The report takes its dataset figures from this table, and the `source` column records which file was checked.

In [22]:
# one row per check
summary = pd.Series(checks, name="value").to_frame()
summary["source"] = path.name

summary.to_csv(TABLES_DIR / "01_sanity_checks.csv")
summary

,value,source
rows,7079081,flights_2024.parquet
first_date,2024-01-01,flights_2024.parquet
last_date,2024-12-31,flights_2024.parquet
months_present,12,flights_2024.parquet
share_cancelled,0.014,flights_2024.parquet
share_diverted,0.002,flights_2024.parquet
completed_without_arr_delay,0,flights_2024.parquet
share_arr_delay_15plus,0.208,flights_2024.parquet
median_arr_delay,-6.000,flights_2024.parquet
mean_arr_delay,7.098,flights_2024.parquet


## 6. Parquet against CSV

We compare the row count and the column totals of both files, to make sure the conversion lost nothing.

In [23]:
# columns to compare
cols = ["arr_delay", "dep_delay", "distance", "cancelled", "diverted"]


if RAW_CSV.exists():
    # totals straight from the CSV, read in chunks so it fits in memory
    n_csv = 0
    sums_csv = pd.Series(0.0, index=cols)
    for chunk in pd.read_csv(RAW_CSV, usecols=cols, chunksize=500_000):
        n_csv += len(chunk)
        sums_csv += chunk.sum()

    # the same totals from the Parquet file
    parquet_cols = load_flights(FLIGHTS_PARQUET, columns=cols)
    sums_parquet = parquet_cols.astype("float64").sum()

    print("rows:", n_csv, "vs", len(parquet_cols))
    display(pd.DataFrame({"csv": sums_csv, "parquet": sums_parquet, "diff": sums_csv - sums_parquet}))

    # all values are whole numbers, so the totals must match exactly
    assert n_csv == len(parquet_cols), "row counts differ"
    assert (sums_csv - sums_parquet).abs().max() == 0, "column totals differ"
else:
    print("Raw CSV not found, so the CSV vs Parquet check is skipped.")

rows: 7079081 vs 7079081


,csv,parquet,diff
arr_delay,"49,441,174.000","49,441,174.000",0.000
dep_delay,"88,563,502.000","88,563,502.000",0.000
distance,"5,903,289,185.000","5,903,289,185.000",0.000
cancelled,"96,315.000","96,315.000",0.000
diverted,"17,499.000","17,499.000",0.000
